In [6]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
from datetime import datetime, timedelta
import time

In [7]:
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

BASE_URL = "https://archive-api.open-meteo.com/v1/archive" #api to get weather data.

end_date = datetime.now().strftime("%Y-%m-%d")
print("end_date is " + end_date)
#start_date = (datetime.now() - timedelta(days=365 * 11)).strftime("%Y-%m-%d")
start_date = "2014-01-01"

end_date is 2025-04-04


In [8]:
locations = [
    {"city": "Calexico-Ethel Street", "zip": "92231", "latitude": 32.676180, "longitude": -115.483070},
    {"city": "Temecula", "zip": "92592", "latitude": 33.583018, "longitude": -117.072202},
    {"city": "Anaheim", "zip": "92805", "latitude": 33.830620, "longitude": -117.938450},
    {"city": "Mira Loma (Van Buren)", "zip": "91752", "latitude": 33.996360, "longitude": -117.492400},
    {"city": "Rubidoux", "zip": "92509", "latitude": 33.999580, "longitude": -117.416010},
    {"city": "Los Angeles-North Main Street", "zip": "90012", "latitude": 34.066590, "longitude": -118.226880},
    {"city": "Thousand Oaks", "zip": "91360", "latitude": 34.210169, "longitude": -118.870509},
    {"city": "Simi Valley-Cochran Street", "zip": "93063", "latitude": 34.276316, "longitude": -118.683685},
    {"city": "Ojai - East Ojai Ave", "zip": "93023", "latitude": 34.448060, "longitude": -119.231300},
    {"city": "Bakersfield-California", "zip": "93301", "latitude": 35.356615, "longitude": -119.062613},
    {"city": "King City 2", "zip": "93930", "latitude": 36.209286, "longitude": -121.126371},
    {"city": "Carmel Valley", "zip": "93924", "latitude": 36.481870, "longitude": -121.733330},
    {"city": "Keeler", "zip": "93530", "latitude": 36.487823, "longitude": -117.871036},
    {"city": "Salinas 3", "zip": "93901", "latitude": 36.694261, "longitude": -121.623271},
    {"city": "Fresno - Garland", "zip": "93728", "latitude": 36.785380, "longitude": -119.773210},
    {"city": "San Jose - Jackson", "zip": "95112", "latitude": 37.348497, "longitude": -121.894898},
    {"city": "Oakland", "zip": "94601", "latitude": 37.743065, "longitude": -122.169935},
    {"city": "Sacramento-Del Paso Manor", "zip": "95815", "latitude": 38.613779, "longitude": -121.368014},
    {"city": "Yuba City", "zip": "95991", "latitude": 39.138773, "longitude": -121.618549},
    {"city": "Chico-East Avenue", "zip": "95926", "latitude": 39.761680, "longitude": -121.840470}
]


In [9]:
def fetch_and_process_weather(location):
    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "pressure_msl", "precipitation", "cloud_cover"],
        "timezone": "auto"
    }
    responses = openmeteo.weather_api(BASE_URL, params=params)
    response = responses[0]  # Process the first response
    hourly = response.Hourly()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s"),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s"),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(2).ValuesAsNumpy(),
        "pressure_msl": hourly.Variables(3).ValuesAsNumpy(),
        "precipitation": hourly.Variables(4).ValuesAsNumpy(),
        "cloud_cover": hourly.Variables(5).ValuesAsNumpy(),
    }

    df = pd.DataFrame(hourly_data)
    df["City"] = location["city"]
    df = df.dropna()
    return df


In [10]:
for location in locations:
    print(f"Fetching data for {location['city']}...")
    try:
        df = fetch_and_process_weather(location)
        filename = f"Data/Weather/historical_weather_data_{location['city'].replace(' ', '_')}.csv"
        df.to_csv(filename, index=False)
        print(f"Data for {location['city']} saved to {filename}")
    except Exception as e:
        print(f"Error fetching data for {location['city']}: {e}")
    time.sleep(20)


Fetching data for Calexico-Ethel Street...
Data for Calexico-Ethel Street saved to Data/Weather/historical_weather_data_Calexico-Ethel_Street.csv
Fetching data for Temecula...
Data for Temecula saved to Data/Weather/historical_weather_data_Temecula.csv
Fetching data for Anaheim...
Data for Anaheim saved to Data/Weather/historical_weather_data_Anaheim.csv
Fetching data for Mira Loma (Van Buren)...
Data for Mira Loma (Van Buren) saved to Data/Weather/historical_weather_data_Mira_Loma_(Van_Buren).csv
Fetching data for Rubidoux...
Data for Rubidoux saved to Data/Weather/historical_weather_data_Rubidoux.csv
Fetching data for Los Angeles-North Main Street...
Data for Los Angeles-North Main Street saved to Data/Weather/historical_weather_data_Los_Angeles-North_Main_Street.csv
Fetching data for Thousand Oaks...
Data for Thousand Oaks saved to Data/Weather/historical_weather_data_Thousand_Oaks.csv
Fetching data for Simi Valley-Cochran Street...
Data for Simi Valley-Cochran Street saved to Data/